In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.semi_supervised import SelfTrainingClassifier
from sklearn.ensemble import RandomForestClassifier
import numpy as np

In [14]:
# Constants
RandomSTATE = 1082
TEST_SIZE = 0.2

In [15]:
# Load CSV
try:
    df = pd.read_csv("../Datasets/csv/spoofing-merged-gps-only.csv")
except FileNotFoundError:
    print("Error: The file '../../Datasets/csv/spoofing-merged-gps-only.csv' was not found.")
    print("Please make sure the path is correct.")
    exit()

In [16]:
# Data Preprocessing

# Clean up column names
df.columns = df.columns.str.replace('[\\[\\]<>]', '_', regex=True)

# Drop label
drop_cols = ['timestamp', 'label']
X = df.drop(columns=drop_cols)
y = df['label']

# Convert categorical labels to numerical
if y.dtype == object or y.dtype.name == 'category':
    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder()
    y = le.fit_transform(y)

# Randomly mask some labels as "unlabeled" to simulate semi-supervised learning
rng = np.random.default_rng(RandomSTATE)
y_semi = y.copy()
mask = rng.choice([True, False], size=y.shape, p=[0.2, 0.8])
y_semi[mask] = -1

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y_semi, test_size=TEST_SIZE, random_state=RandomSTATE, stratify=y)

In [17]:
base_clf = RandomForestClassifier(random_state=RandomSTATE)
self_training_model = SelfTrainingClassifier(
    base_clf,
    criterion='k_best',
    k_best=5,  # Number of features to select
)
self_training_model.fit(X_train, y_train)

SelfTrainingClassifier(base_estimator=RandomForestClassifier(random_state=1082),
                       criterion='k_best', k_best=5)

In [18]:
y_pred = self_training_model.predict(X_test)

labeled_indices = (y_test != -1)

# Only evaluate on the labeled samples in the test set
accuracy = accuracy_score(y_test[labeled_indices], y_pred[labeled_indices])
print(f"Model Accuracy on Test Set: {accuracy * 100:.2f}%")
print("Confusion Matrix:")
print(confusion_matrix(y_test[labeled_indices], y_pred[labeled_indices]))
print("\nClassification Report:")
print(classification_report(y_test[labeled_indices], y_pred[labeled_indices], zero_division=0))

Model Accuracy on Test Set: 100.00%
Confusion Matrix:
[[520   0]
 [  0  80]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       520
           1       1.00      1.00      1.00        80

    accuracy                           1.00       600
   macro avg       1.00      1.00      1.00       600
weighted avg       1.00      1.00      1.00       600

